# Neural Machine Translation with Attention mechanism

### What is Attention?

Attention is an interface between the encoder and decoder that provides the decoder with information from every encoder hidden state. With this setting, the model is able to selectively focus on useful parts of the input sequence and hence, learn the alignment between them. This helps the model to cope effectively with long input sentences .

In [10]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [11]:
!pip install chart_studio

In [12]:
import pandas as pd 
import tensorflow as tf

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import unicodedata
import re
import numpy as np
import time
import string

import chart_studio.plotly
import chart_studio.plotly as py
from plotly.offline import init_notebook_mode, iplot
import plotly.graph_objs as go


### As in case of any NLP task, after reading the input file, we perform the basic cleaning and preprocessing as follows:

In [13]:
file_path = '/home/asr/Documents/nmt/dataset/Seq2Seq_NMT.txt' # please set the path according to your system

In [14]:
lines = open(file_path, encoding='UTF-8').read().strip().split('\n')
lines[5000:5010]

['ངེའི་ ཚོས་གཞི་ དགའ་ཤོས་ དེ་ ཧོནམོ་ ཨིན \tmy favorite color is blue',
 'ཁྱོད་ ཀྱི་ ཆུ་དུང་ འདི་ བཅོ་ མ་ ཚུགས་ པ་ ཅིན་ ང་བཅས་ ཀྱིས་ ཆུ་དུང་ བཅོ་ མི་ བོ་ དགོ \tif you cannot fix the pipe  we will have to call a plumber',
 'ཇོ་སེཕ་ དང་ གཅིག་ཁར་ མོ་ ལུ་ ཨ་ལོ་ གསུམ་ སྡོད་ ནུག \twith joseph  she has five children',
 'ཁྱོད་ ཧ་ལས་སི་སི་ འདུག \tyou are wonderful',
 'ཁྲོམ་ལམ་ གི་ མཇུག་ ལུ་ གཡས་ ཁ་ཐུག་ བསྒྱིར \tturn right at the end of that street',
 'ཁྱོད་ ཀྱིས་ དེ་ བཅུ་གཉིས་ ཚན་པ་ སྦེ་ མཁོ་མངག་ འབད་བ་ཅིན་ ཁེ་ཏོག་ཏོ་ ཡོདཔ་ ཨིན \tit is cheaper if you order these by the dozen',
 'མི་ དག་པ་ཅིག་ ལུ་ མཛུབ་གནོན་ ལྕགས་ པར་ འདུག \tfew people have typewriters',
 'ང་བཅས་ སྡེ་ཚན་ ལེགས་ཤོམ་ ཅིག་ ཨིན \twe make a good team',
 'སར་ ར་ ནུ་ འདི་ མིང་གཏམ་ཅན་ གྱི་ རྩོམ་པ་པོ་ དང་ གསལ་བཤད་པ་ ཡང་ ཨིན \tsir ranu is also a renowned author and speaker',
 'ང་ མཐའ་འཁོར་ ལུ་ དོ་རུང་ ཚར་ གཅིག་ བལྟ་ དགོ་ པས \ti want to take another look around']

In [15]:
print("total number of records: ",len(lines))

total number of records:  232489


In [16]:
exclude = set(string.punctuation) # Set of all special characters
remove_digits = str.maketrans('', '', string.digits) # Set of all digits

### Function to preprocess English sentence

In [17]:
def preprocess_eng_sentence(sent):
    '''Function to preprocess English sentence'''
    sent = sent.lower() # lower casing
    sent = re.sub("'", '', sent) # remove the quotation marks if any
    sent = ''.join(ch for ch in sent if ch not in exclude)
    sent = sent.translate(remove_digits) # remove the digits
    sent = sent.strip()
    sent = re.sub(" +", " ", sent) # remove extra spaces
    sent = '<start> ' + sent + ' <end>' # add <start> and <end> tokens
    return sent

### Function to preprocess Dzongkha sentence

In [18]:
def preprocess_dzo_sentence(sent):
    '''Function to preprocess Dzongkha sentence'''
    sent = re.sub("'", '', sent) # remove the single quotation marks if any
    sent = re.sub("[༠༡༢༣༤༥༦༧༨༩]", "", sent) # remove the digits
    sent = re.sub(r'་(?=\s)', ' ', sent)#remove the TSHA the dzongkha word dilimiter
    sent = re.sub(r'་\s*[& ]', " ",sent)
    sent=re.sub(r"\s*།\s*$", "", sent)#remove the SHED the dznogkha sentence dilimiter
    sent=re.sub(r'\s#\s'," ",sent)
    sent = sent.strip()
    sent = re.sub(" +", " ", sent) # remove extra spaces
    sent = '<start> ' + sent + ' <end>' # add <start> and <end> tokens
    return sent

### Generate pairs of cleaned English and Dzongkha sentences with start and end tokens added.

In [20]:
# Generate pairs of cleaned English and Dzongkha sentences
sent_pairs = []
for line in lines:
    sent_pair = []
    eng = line.rstrip().split('\t')[1]
    dzo = line.rstrip().split('\t')[0]
    eng = preprocess_eng_sentence(eng)
    sent_pair.append(eng)
    dzo = preprocess_dzo_sentence(dzo)
    sent_pair.append(dzo)
    sent_pairs.append(sent_pair)
sent_pairs[5000:5010]

[['<start> my favorite color is blue <end>',
  '<start> ངེའི ཚོས་གཞི དགའ་ཤོས དེ ཧོནམོ ཨིན <end>'],
 ['<start> if you cannot fix the pipe we will have to call a plumber <end>',
  '<start> ཁྱོད ཀྱི ཆུ་དུང འདི བཅོ མ ཚུགས པ ཅིན ང་བཅས ཀྱིས ཆུ་དུང བཅོ མི བོ དགོ <end>'],
 ['<start> with joseph she has five children <end>',
  '<start> ཇོ་སེཕ དང གཅིག་ཁར མོ ལུ ཨ་ལོ གསུམ སྡོད ནུག <end>'],
 ['<start> you are wonderful <end>', '<start> ཁྱོད ཧ་ལས་སི་སི འདུག <end>'],
 ['<start> turn right at the end of that street <end>',
  '<start> ཁྲོམ་ལམ གི མཇུག ལུ གཡས ཁ་ཐུག བསྒྱིར <end>'],
 ['<start> it is cheaper if you order these by the dozen <end>',
  '<start> ཁྱོད ཀྱིས དེ བཅུ་གཉིས ཚན་པ སྦེ མཁོ་མངག འབད་བ་ཅིན ཁེ་ཏོག་ཏོ ཡོདཔ ཨིན <end>'],
 ['<start> few people have typewriters <end>',
  '<start> མི དག་པ་ཅིག ལུ མཛུབ་གནོན ལྕགས པར འདུག <end>'],
 ['<start> we make a good team <end>',
  '<start> ང་བཅས སྡེ་ཚན ལེགས་ཤོམ ཅིག ཨིན <end>'],
 ['<start> sir ranu is also a renowned author and speaker <end>',
  '<start> སར ར ནུ

In [21]:
# This class creates a word -> index mapping (e.g,. "dad" -> 5) and vice-versa 
# (e.g., 5 -> "dad") for each language,
class LanguageIndex():
    def __init__(self, lang):
        self.lang = lang
        self.word2idx = {}
        self.idx2word = {}
        self.vocab = set()

        self.create_index()

    def create_index(self):
        for phrase in self.lang:
            self.vocab.update(phrase.split(' '))

        self.vocab = sorted(self.vocab)

        self.word2idx['<pad>'] = 0
        for index, word in enumerate(self.vocab):
            self.word2idx[word] = index + 1

        for word, index in self.word2idx.items():
            self.idx2word[index] = word

In [22]:
def max_length(tensor):
    return max(len(t) for t in tensor)

### Tokenization and Padding

In [23]:
def load_dataset(pairs, num_examples):
    # pairs => already created cleaned input, output pairs

    # index language using the class defined above    
    inp_lang = LanguageIndex(en for en, ma in pairs)
    targ_lang = LanguageIndex(ma for en, ma in pairs)
    
    # Vectorize the input and target languages
    
    # English sentences
    input_tensor = [[inp_lang.word2idx[s] for s in en.split(' ')] for en, ma in pairs]
    
    # Marathi sentences
    target_tensor = [[targ_lang.word2idx[s] for s in ma.split(' ')] for en, ma in pairs]
    
    # Calculate max_length of input and output tensor
    # Here, we'll set those to the longest sentence in the dataset
    max_length_inp, max_length_tar = max_length(input_tensor), max_length(target_tensor)
    
    # Padding the input and output tensor to the maximum length
    input_tensor = tf.keras.preprocessing.sequence.pad_sequences(input_tensor, maxlen=max_length_inp,padding='post')
    
    target_tensor = tf.keras.preprocessing.sequence.pad_sequences(target_tensor, maxlen=max_length_tar,padding='post')
    
    return input_tensor, target_tensor, inp_lang, targ_lang, max_length_inp, max_length_tar

In [24]:
input_tensor, target_tensor, inp_lang, targ_lang, max_length_inp, max_length_targ = load_dataset(sent_pairs, len(lines))

### Creating training and validation sets using an 80-20 split

In [25]:
# Creating training and validation sets using an 80-20 split
input_tensor_train, input_tensor_val, target_tensor_train, target_tensor_val = train_test_split(input_tensor, target_tensor, test_size=0.1, random_state = 101)

# Show length
len(input_tensor_train), len(target_tensor_train), len(input_tensor_val), len(target_tensor_val)

(209240, 209240, 23249, 23249)

In [26]:
BUFFER_SIZE = len(input_tensor_train)
BATCH_SIZE = 64
N_BATCH = BUFFER_SIZE//BATCH_SIZE
embedding_dim = 256
units = 1024
vocab_inp_size = len(inp_lang.word2idx)
vocab_tar_size = len(targ_lang.word2idx)

dataset = tf.data.Dataset.from_tensor_slices((input_tensor_train, target_tensor_train)).shuffle(BUFFER_SIZE)
dataset = dataset.batch(BATCH_SIZE, drop_remainder=True)

2023-03-24 01:19:27.362133: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-03-24 01:19:27.363545: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-03-24 01:19:27.365081: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcusolver.so.11'; dlerror: libcusolver.so.11: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: :/home/asr/anaconda3/envs/gpu2/lib/libcudart.so.11.0
2023-03-24 01:19:27.365169: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1934] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to us

We'll be using GRUs instead of LSTMs as we only have to create one state and implementation would be easier.

### Create GRU units

In [27]:
def gru(units):

    return tf.keras.layers.GRU(units, 
                                   return_sequences=True, 
                                   return_state=True, 
                                   recurrent_activation='sigmoid', 
                                   recurrent_initializer='glorot_uniform')


### The next step is to define the encoder and decoder network.

The input to the encoder will be the sentence in English and the output will be the hidden state and cell state of the GRU.

In [28]:
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, enc_units, batch_sz):
        super(Encoder, self).__init__()
        self.batch_sz = batch_sz
        self.enc_units = enc_units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.gru = gru(self.enc_units)
        
    def call(self, x, hidden):
        x = self.embedding(x)
        output, state = self.gru(x, initial_state = hidden)        
        return output, state
    
    def initialize_hidden_state(self):
        return tf.zeros((self.batch_sz, self.enc_units))

The next step is to define the decoder. The decoder will have two inputs: the hidden state and cell state from the encoder and the input sentence, which actually will be the output sentence with a token appended at the beginning.

In [29]:
class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, dec_units, batch_sz):
        super(Decoder, self).__init__()
        self.batch_sz = batch_sz
        self.dec_units = dec_units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.gru = gru(self.dec_units)
        self.fc = tf.keras.layers.Dense(vocab_size)
        
        # used for attention
        self.W1 = tf.keras.layers.Dense(self.dec_units)
        self.W2 = tf.keras.layers.Dense(self.dec_units)
        self.V = tf.keras.layers.Dense(1)
        
    def call(self, x, hidden, enc_output):

        hidden_with_time_axis = tf.expand_dims(hidden, 1)
        
        # score shape == (batch_size, max_length, 1)
        # we get 1 at the last axis because we are applying tanh(FC(EO) + FC(H)) to self.V
        score = self.V(tf.nn.tanh(self.W1(enc_output) + self.W2(hidden_with_time_axis)))
        
        # attention_weights shape == (batch_size, max_length, 1)
        attention_weights = tf.nn.softmax(score, axis=1)
        
        # context_vector shape after sum == (batch_size, hidden_size)
        context_vector = attention_weights * enc_output
        context_vector = tf.reduce_sum(context_vector, axis=1)
        
        # x shape after passing through embedding == (batch_size, 1, embedding_dim)
        x = self.embedding(x)
        
        # x shape after concatenation == (batch_size, 1, embedding_dim + hidden_size)
        x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)
        
        # passing the concatenated vector to the GRU
        output, state = self.gru(x)
        
        # output shape == (batch_size * 1, hidden_size)
        output = tf.reshape(output, (-1, output.shape[2]))
        
        # output shape == (batch_size * 1, vocab)
        x = self.fc(output)
        
        return x, state, attention_weights
        
    def initialize_hidden_state(self):
        return tf.zeros((self.batch_sz, self.dec_units))

Create encoder and decoder objects from their respective classes.

In [33]:
encoder = Encoder(vocab_inp_size, embedding_dim, units, BATCH_SIZE)
decoder = Decoder(vocab_tar_size, embedding_dim, units, BATCH_SIZE)

### Define the optimizer and the loss function.

In [34]:
optimizer = tf.optimizers.Adam()

def loss_function(real, pred):
    mask = 1 - np.equal(real, 0)
    loss_ = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=real, logits=pred) * mask
    return tf.reduce_mean(loss_)

In [35]:
import os
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(optimizer=optimizer,encoder=encoder,decoder=decoder)

### Training the Model

In [36]:
EPOCHS = 5

for epoch in range(EPOCHS):
    start = time.time()
    
    hidden = encoder.initialize_hidden_state()
    total_loss = 0
    
    for (batch, (inp, targ)) in enumerate(dataset):
        loss = 0
        
        with tf.GradientTape() as tape:
            enc_output, enc_hidden = encoder(inp, hidden)
            
            dec_hidden = enc_hidden
            
            dec_input = tf.expand_dims([targ_lang.word2idx['<start>']] * BATCH_SIZE, 1)       
            
            # Teacher forcing - feeding the target as the next input
            for t in range(1, targ.shape[1]):
                # passing enc_output to the decoder
                predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)
                
                loss += loss_function(targ[:, t], predictions)
                
                # using teacher forcing
                dec_input = tf.expand_dims(targ[:, t], 1)
        
        batch_loss = (loss / int(targ.shape[1]))
        
        total_loss += batch_loss
        
        variables = encoder.variables + decoder.variables
        
        gradients = tape.gradient(loss, variables)
        
        optimizer.apply_gradients(zip(gradients, variables))
        
        if batch % 100 == 0:
            print('Epoch {} Batch {} Loss {:.4f}'.format(epoch + 1,batch,batch_loss.numpy()))
    # saving (checkpoint) the model every epoch
    checkpoint.save(file_prefix = checkpoint_prefix)
    
    print('Epoch {} Loss {:.4f}'.format(epoch + 1,total_loss / N_BATCH))
    print('Time taken for 1 epoch {} sec\n'.format(time.time() - start))


Epoch 1 Batch 0 Loss 4.6574
Epoch 1 Batch 100 Loss 2.6272
Epoch 1 Batch 200 Loss 2.2393
Epoch 1 Batch 300 Loss 2.1865
Epoch 1 Batch 400 Loss 2.1062
Epoch 1 Batch 500 Loss 1.9988
Epoch 1 Batch 600 Loss 1.9718
Epoch 1 Batch 700 Loss 1.9498
Epoch 1 Batch 800 Loss 1.9101
Epoch 1 Batch 900 Loss 1.8189
Epoch 1 Batch 1000 Loss 1.7958
Epoch 1 Batch 1100 Loss 1.9699
Epoch 1 Batch 1200 Loss 2.0705
Epoch 1 Batch 1300 Loss 1.7051
Epoch 1 Batch 1400 Loss 1.7806
Epoch 1 Batch 1500 Loss 1.6049
Epoch 1 Batch 1600 Loss 1.7192
Epoch 1 Batch 1700 Loss 1.6822
Epoch 1 Batch 1800 Loss 1.6019
Epoch 1 Batch 1900 Loss 1.6133
Epoch 1 Batch 2000 Loss 1.6384
Epoch 1 Batch 2100 Loss 1.5686
Epoch 1 Batch 2200 Loss 1.6205
Epoch 1 Batch 2300 Loss 1.6493
Epoch 1 Batch 2400 Loss 1.5577
Epoch 1 Batch 2500 Loss 1.4226
Epoch 1 Batch 2600 Loss 1.5524
Epoch 1 Batch 2700 Loss 1.5671
Epoch 1 Batch 2800 Loss 1.5142
Epoch 1 Batch 2900 Loss 1.4956
Epoch 1 Batch 3000 Loss 1.4765
Epoch 1 Batch 3100 Loss 1.3765
Epoch 1 Batch 3200 L

### Restoring the latest checkpoint

In [37]:
# restoring the latest checkpoint in checkpoint_dir
checkpoint.restore(tf.train.latest_checkpoint(checkpoint_dir))

### Inference setup and testing:

In [38]:
def evaluate(inputs, encoder, decoder, inp_lang, targ_lang, max_length_inp, max_length_targ):
    
    attention_plot = np.zeros((max_length_targ, max_length_inp))
    sentence = ''
    for i in inputs[0]:
        if i == 0:
            break
        sentence = sentence + inp_lang.idx2word[i] + ' '
    sentence = sentence[:-1]
    
    inputs = tf.convert_to_tensor(inputs)
    
    result = ''

    hidden = [tf.zeros((1, units))]
    enc_out, enc_hidden = encoder(inputs, hidden)

    dec_hidden = enc_hidden
    dec_input = tf.expand_dims([targ_lang.word2idx['<start>']], 0)

    for t in range(max_length_targ):
        predictions, dec_hidden, attention_weights = decoder(dec_input, dec_hidden, enc_out)
        
        # storing the attention weights to plot later on
        attention_weights = tf.reshape(attention_weights, (-1, ))
        attention_plot[t] = attention_weights.numpy()

        predicted_id = tf.argmax(predictions[0]).numpy()

        result += targ_lang.idx2word[predicted_id] + ' '

        if targ_lang.idx2word[predicted_id] == '<end>':
            return result, sentence, attention_plot
        
        # the predicted ID is fed back into the model
        dec_input = tf.expand_dims([predicted_id], 0)

    return result, sentence, attention_plot


### Function to predict (translate) a randomly selected test point

In [39]:
def predict_random_val_sentence():
    actual_sent = ''
    k = np.random.randint(len(input_tensor_val))
    random_input = input_tensor_val[k]
    random_output = target_tensor_val[k]
    random_input = np.expand_dims(random_input,0)
    result, sentence, attention_plot = evaluate(random_input, encoder, decoder, inp_lang, targ_lang, max_length_inp, max_length_targ)
    print('Input: {}'.format(sentence[8:-6]))
    print('Predicted translation: {}'.format(result[:-6]))
    for i in random_output:
        if i == 0:
            break
        actual_sent = actual_sent + targ_lang.idx2word[i] + ' '
    actual_sent = actual_sent[8:-7]
    print('Actual translation: {}'.format(actual_sent))
    attention_plot = attention_plot[:len(result.split(' '))-2, 1:len(sentence.split(' '))-1]
    sentence, result = sentence.split(' '), result.split(' ')
    sentence = sentence[1:-1]
    result = result[:-2]

    # use plotly to generate the heat map
    trace = go.Heatmap(z = attention_plot, x = sentence, y = result, colorscale='greens')
    data=[trace]
    iplot(data)

In [59]:
predict_random_val_sentence()

Input: we cannot get close to the enemy
Predicted translation: ང་བཅས དགྲ་བྱང་ཕྱད ཀྱི སྦོ་ལོགས་ཁར ལས ཐར མི ཚུགས 
Actual translation: ང་བཅས ཀྱིས དགྲ གི སྦོ་ལོགས་ཁར འགྱོ མི རུང


In [54]:
checkpoint.save("Attention_weight.h5")

'Attention_weight.h5-6'